<a href="https://colab.research.google.com/github/ian-menachery/ECON3916-Statistical-Machine-Learning/blob/main/%5BLab_13%5D_Hedonic_Pricing_and_the_FWL_Theorem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

# Step 1: Ingestion and Naive Model
url = 'Zillow_California_2026_Hedonic.csv'
df = pd.read_csv(url)

naive_model = smf.ols('Sale_Price ~ Property_Age', data=df).fit()
print(naive_model.summary())
print("\nNaive Age Coefficient:", naive_model.params['Property_Age'])

# Step 2: The Multivariate Model
multi_model = smf.ols('Sale_Price ~ Property_Age + Distance_to_Tech_Hub', data=df).fit()
print(multi_model.summary())
print("\nMultivariate Age Coefficient:", multi_model.params['Property_Age'])

# 3a: Partial out distance from Price
res_y_model = smf.ols('Sale_Price ~ Distance_to_Tech_Hub', data=df).fit()
df['Price_Residuals'] = res_y_model.resid

# 3b: Partial out distance from Age
res_x_model = smf.ols('Property_Age ~ Distance_to_Tech_Hub', data=df).fit()
df['Age_Residuals'] = res_x_model.resid

# 3c: Regress Residuals on Residuals (-1 removes the intercept for exact mathematical matching)
fwl_model = smf.ols('Price_Residuals ~ Age_Residuals - 1', data=df).fit()
print("\nFWL Isolated Age Coefficient:", fwl_model.params['Age_Residuals'])

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.757
Model:                            OLS   Adj. R-squared:                  0.757
Method:                 Least Squares   F-statistic:                     3105.
Date:                Wed, 18 Mar 2026   Prob (F-statistic):          1.26e-308
Time:                        19:39:19   Log-Likelihood:                -12818.
No. Observations:                1000   AIC:                         2.564e+04
Df Residuals:                     998   BIC:                         2.565e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept     3.013e+05   7218.570     41.742   

In [3]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import plotly.graph_objects as go

# ---------------------------------------------------------
# 1. Synthetic Data Generation (For standalone execution)
# ---------------------------------------------------------
np.random.seed(42)
n_samples = 200

# Simulating the confounding environment:
# Older homes are closer to tech hubs (negative correlation)
distance = np.random.uniform(1, 50, n_samples)
# Age is inversely related to distance, plus some noise
age = 100 - (1.5 * distance) + np.random.normal(0, 10, n_samples)
age = np.clip(age, 0, 100) # Ensure no negative ages

# True underlying pricing mechanism:
# Price drops by $5k per year of age, drops by $10k per mile from hub
base_price = 1000000
price = base_price - (5000 * age) - (10000 * distance) + np.random.normal(0, 50000, n_samples)

df = pd.DataFrame({
    'Sale_Price': price,
    'Property_Age': age,
    'Distance_to_Tech_Hub': distance
})

# ---------------------------------------------------------
# 2. Fit the Multivariate OLS Model
# ---------------------------------------------------------
multi_model = smf.ols('Sale_Price ~ Property_Age + Distance_to_Tech_Hub', data=df).fit()

# ---------------------------------------------------------
# 3. Generate the 3D Regression Plane (The Meshgrid)
# ---------------------------------------------------------
# MECHANISM CHECK: Creating the meshgrid
# We need to define the 2D floor (X and Y axes) to plot the 3D ceiling (Z axis/Hyperplane).
# We extract the min and max values of our two independent variables to define the grid boundaries.
x_range = np.linspace(df['Property_Age'].min(), df['Property_Age'].max(), 20)
y_range = np.linspace(df['Distance_to_Tech_Hub'].min(), df['Distance_to_Tech_Hub'].max(), 20)

# np.meshgrid creates a 2D coordinate matrix from our 1D arrays.
# xx and yy now contain all possible combinations of Age and Distance within our range.
xx, yy = np.meshgrid(x_range, y_range)

# MECHANISM CHECK: Extracting coefficients to calculate Z (Predictions)
# We pull the intercept (\beta_0), Age coefficient (\beta_1), and Distance coefficient (\beta_2)
b0 = multi_model.params['Intercept']
b1 = multi_model.params['Property_Age']
b2 = multi_model.params['Distance_to_Tech_Hub']

# We calculate the predicted Sale_Price (Z) for every coordinate pair on our meshgrid.
# This strictly follows the linear equation: Z = b0 + (b1 * X) + (b2 * Y)
zz = b0 + (b1 * xx) + (b2 * yy)

# ---------------------------------------------------------
# 4. Plotly 3D Visualization
# ---------------------------------------------------------
fig = go.Figure()

# Add the empirical data points (The Scatter)
fig.add_trace(go.Scatter3d(
    x=df['Property_Age'],
    y=df['Distance_to_Tech_Hub'],
    z=df['Sale_Price'],
    mode='markers',
    marker=dict(size=4, color='blue', opacity=0.6),
    name='Observed Data'
))

# Add the fitted regression plane (The Surface)
fig.add_trace(go.Surface(
    x=xx,
    y=yy,
    z=zz,
    colorscale='Viridis',
    opacity=0.8,
    name='Regression Plane',
    showscale=False
))

# Update layout for presentation
fig.update_layout(
    title='Multivariate OLS: Sale Price Hyperplane',
    scene=dict(
        xaxis_title='Property Age (Years)',
        yaxis_title='Distance to Tech Hub (Miles)',
        zaxis_title='Sale Price ($)',
    ),
    width=900,
    height=700,
    margin=dict(l=0, r=0, b=0, t=40)
)

fig.show()